In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma, FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser




In [77]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "YyCXp_E4sMU"

api = YouTubeTranscriptApi()

# Change .fetch() to .get_transcript()
transcript_list = api.fetch(video_id, languages=['en']).to_raw_data()

# Join the text fragments
transcript = " ".join([entry["text"] for entry in transcript_list])

print(transcript)

A very good evening to you ladies and gentlemen. We can do better. A very good evening to you ladies and gentlemen. Oh wow. It's an absolute joy, honor and pleasure to be here this evening. Let me kick this evening off with a favorite story of mine. There was a little boy about 6 years old who came to a balloon seller who was selling balloons on the street and he asked the balloon seller, "Uncle, will this red balloon go up in the air?" The balloon seller said, "Of course it will." Said, "And the blue one?" 100% it will. The pink one, most certainly it will. And you know how kids can be. They can ask funny questions and I give funny answers as well. A 4-year-old once came to a shopkeeper and said to the shopkeeper girl, he said, "Uncle, fouryear-old girl." The shopkeeper said, he said uncle. The shopkeeper said, he said uncle children very innocent, charming, clever, smart. They can ask funny questions and they can give funny answers too. A teacher in class asked a six-year-old which s

In [78]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 200)
docs = splitter.create_documents([transcript])
print(docs[1])

page_content='to a balloon seller who was selling balloons on the street and he asked the balloon seller, "Uncle, will this red balloon go up in the air?" The balloon seller said, "Of course it will." Said, "And the blue one?" 100% it will. The pink one, most certainly it will. And you know how kids can be. They can ask funny questions and I give funny answers as well. A 4-year-old once came to a shopkeeper and said to the shopkeeper girl, he said, "Uncle, fouryear-old girl." The shopkeeper said, he said'


In [79]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(documents = docs, embedding = embedding, collection_name="youtube_transcript_1")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3434.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [80]:

retreiver = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})




In [92]:


youtube_template = """
You are a Video Research Assistant. Your goal is to answer questions based strictly on the provided YouTube transcript segments.

---
TRANSCRIPT CONTEXT:
{docs}
---

USER QUESTION: {query}

STRICT GUIDELINES:

1. Use ONLY the transcript context above. Do not use outside facts.
2. If the context doesn't mention the answer, say: "I'm sorry, that wasn't mentioned in this video."
3. If the transcript is unstructured (missing punctuation), interpret the flow of speech to provide a coherent answer.
4. Mention the specific part of the video if the context allows.
5.Answer in just 2 sentences maximum.

DETAILED RESPONSE:
"""

prompt_1 = PromptTemplate(
    template=youtube_template, 
    input_variables=["docs", "query"]
)



In [93]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [83]:
Final_prompt = prompt_1.invoke({"query": ques, "docs": context_text})
print(Final_prompt)


text='\nYou are a Video Research Assistant. Your goal is to answer questions based strictly on the provided YouTube transcript segments.\n\n---\nTRANSCRIPT CONTEXT:\nA very good evening to you ladies and gentlemen. We can do better. A very good evening to you ladies and gentlemen. Oh wow. It\'s an absolute joy, honor and pleasure to be here this evening. Let me kick this evening off with a favorite story of mine. There was a little boy about 6 years old who came to a balloon seller who was selling balloons on the street and he asked the balloon seller, "Uncle, will this red balloon go up in the air?" The balloon seller said, "Of course it will." Said, "And the blue one?" 100% it will. The pink one, most certainly it will. And you know how kids can be. They can ask funny questions and I give funny answers as well. A 4-year-old once came to a shopkeeper and said to the shopkeeper girl, he said, "Uncle, fouryear-old girl." The shopkeeper said, he said uncle. The shopkeeper said, he said u

In [84]:
llm = HuggingFaceEndpoint(
   repo_id="meta-llama/Llama-3.2-1B-Instruct",
   task="text2text-generation",
)

model = ChatHuggingFace(llm=llm, temperature=0.7)

response = model.invoke(Final_prompt)
print(response.text)

Based on the provided YouTube transcript segment, the story of the six-year-old girl revolves around her asking the shopkeeper or the class teacher if the balloons will go up in the air when filled with a specific color of gas.

The context states: "A little boy about 6 years old who came to a balloon seller who was selling balloons on the street and he asked the balloon seller, 'Uncle, will this red balloon go up in the air?' The balloon seller said, "Of course it will." Said, 'And the blue one?' 100% it will. The pink one, most certainly it will. And you know how kids can be. They can ask funny questions and I give funny answers as well."

This question was asked by a six-year-old girl, and the balloon seller responded, "Look, darling." The balloons don't go up in the air based on their colors, but rather based on what's filled inside of them, the gas, the air.


In [99]:
parallel_chain = RunnableParallel({
    "docs": retreiver | RunnableLambda(format_docs),
    "query": RunnablePassthrough()
})



parser = StrOutputParser()


final_chain = parallel_chain | prompt_1 | model | parser
final_response = final_chain.invoke("what is the video about?")
print(final_response)

The video is about John Lennon's message to a boy, encouraging him to remember that the journey to happiness and success is just as important as the destination. The boy's question about how many people had failed in an exam, to which John Lennon replied, "You didn't get life right," emphasizing the importance of enjoying the journey and being happy, regardless of the outcome.
